# NB-01: RMSNorm, Sinusoidal Embeddings, and the Modulate Function

## Learning Objectives
- Understand why WAN 2.2 uses RMSNorm instead of LayerNorm -- no bias, no mean centering, float32 upcast for numerical stability
- Trace how `sinusoidal_embedding_1d` converts integer timesteps into continuous frequency-domain vectors using float64 precision
- See how the `modulate` function applies scale and shift conditioning, and verify that scale=0, shift=0 produces an identity transform

## Prerequisites
- **Prior notebooks:** None (this is the first notebook in the series)
- **Assumed concepts:** PyTorch `nn.Module`, forward pass mechanics, tensor shapes (B, S, dim notation), basic understanding of normalization in neural networks

## Concept Map
- **RMSNorm** -> used in SelfAttention q/k normalization (NB-04) and DiT block pre-norms (NB-06 in Track 2)
- **sinusoidal_embedding_1d** -> used in `WanModel.time_embedding` to encode diffusion timesteps (NB-08 in Track 2)
- **modulate** -> used in adaLN-Zero conditioning (NB-05) and DiTBlock forward pass (NB-06 in Track 2)

In [1]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import the three symbols this notebook covers
from diffsynth.models.wan_video_dit import RMSNorm, sinusoidal_embedding_1d, modulate

import torch
import torch.nn as nn

print("Setup complete.")

Project root: /home/basheer/Signapse/Codes/wan22-architecture-course
Setup complete.


## 1. RMSNorm

RMSNorm (Root Mean Square Normalization) is the normalization layer used throughout WAN 2.2's DiT blocks. It differs from the more familiar LayerNorm in three key ways:

1. **No mean centering**: LayerNorm subtracts the mean before normalizing. RMSNorm skips that step -- it only divides by the root mean square. This makes it slightly faster and avoids centering the activations.

2. **No bias parameter**: LayerNorm has both a learnable `weight` (gamma) and a learnable `bias` (beta), giving `2 * dim` parameters. RMSNorm has only `weight`, giving `dim` parameters. The bias is zero by construction.

3. **Float32 upcast for stability**: Before computing the RMS, WAN 2.2's RMSNorm upcasts the input to `float32` regardless of the input dtype (which is typically `bfloat16` during training). After normalizing, it casts back. This prevents underflow or overflow in the squared-mean computation.

In production, the base model uses `dim=1536` and S2V uses `dim=5120`. Our reduced config uses `dim=384` for CPU demos.

The VACE encoder reuses this same RMSNorm implementation for its control pathway normalization -- covered in Track 4.

Source: `diffsynth/models/wan_video_dit.py`, line 101

In [2]:
# Source: diffsynth/models/wan_video_dit.py, line 101
# Annotated walkthrough of the RMSNorm implementation:
#
# class RMSNorm(nn.Module):
#     def __init__(self, dim, eps=1e-5):
#         super().__init__()
#         self.eps = eps                                    # small constant for numerical stability
#         self.weight = nn.Parameter(torch.ones(dim))       # learnable scale (no bias)
#
#     def norm(self, x):
#         # x.pow(2).mean(dim=-1, keepdim=True)  -> RMS^2 per token: [B, S, 1]
#         # torch.rsqrt(...) -> 1/sqrt(RMS^2 + eps): [B, S, 1]
#         # x * ...          -> normalized: [B, S, dim]
#         return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
#
#     def forward(self, x):
#         dtype = x.dtype               # remember original dtype (e.g., bfloat16)
#         # norm(x.float()) upcasts to float32, then casts back
#         return self.norm(x.float()).to(dtype) * self.weight  # [B, S, dim]

print("RMSNorm walkthrough: weight-only normalization by root mean square")
print("Key: forward() upcasts to float32 for stability, then casts back to input dtype")

RMSNorm walkthrough: weight-only normalization by root mean square
Key: forward() upcasts to float32 for stability, then casts back to input dtype


In [3]:
# Reduced config (validated for S2V divisibility: head_dim//2 % 3 == 0)
B, S, dim = 1, 10, 384
x = torch.randn(B, S, dim)
norm = RMSNorm(dim=dim)
out = norm(x)
assert out.shape == torch.Size([B, S, dim]), f"Expected {(B, S, dim)}, got {out.shape}"
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Parameters: weight {norm.weight.shape} (no bias)")
print(f"Parameter count: {sum(p.numel() for p in norm.parameters())}")

Input shape:  torch.Size([1, 10, 384])
Output shape: torch.Size([1, 10, 384])
Parameters: weight torch.Size([384]) (no bias)
Parameter count: 384


### RMSNorm vs LayerNorm

| Property | RMSNorm | LayerNorm |
|----------|---------|----------|
| Mean centering | No | Yes (subtracts mean) |
| Bias parameter | No | Yes |
| Normalization | rsqrt of mean-squared | Subtract mean, divide by std |
| Speed | Faster (fewer ops) | Baseline |
| Parameters per layer | `dim` | `2 * dim` |

For `dim=384` (reduced config), RMSNorm has 384 parameters vs LayerNorm's 768. Both produce the same output shape `[B, S, dim]`.

In [4]:
ln = nn.LayerNorm(dim)
out_ln = ln(x)
assert out_ln.shape == torch.Size([B, S, dim])
print(f"RMSNorm params: {sum(p.numel() for p in norm.parameters()):,}")
print(f"LayerNorm params: {sum(p.numel() for p in ln.parameters()):,}")
assert out.shape == out_ln.shape, "Both must produce same output shape"
print(f"Both produce shape: {out.shape}")
print(f"RMSNorm weight shape: {norm.weight.shape}")
print(f"LayerNorm weight shape: {ln.weight.shape}, bias shape: {ln.bias.shape}")

RMSNorm params: 384
LayerNorm params: 768
Both produce shape: torch.Size([1, 10, 384])
RMSNorm weight shape: torch.Size([384])
LayerNorm weight shape: torch.Size([384]), bias shape: torch.Size([384])


## 2. Sinusoidal Timestep Embeddings

The `sinusoidal_embedding_1d` function converts integer timestep positions (e.g., diffusion step 0, 1, 2, ..., 999) into dense continuous vectors using sinusoidal functions. This is the same idea as the classic transformer positional encoding but applied to diffusion timesteps.

**How it works:**
1. For each timestep position `t`, compute a set of frequencies: `10000^(-k / (dim//2))` for `k = 0, 1, ..., dim//2 - 1`
2. Multiply each position by its frequency band: `t * freq_k` (via `torch.outer`)
3. Apply cosine to the first half of dimensions, sine to the second half
4. Concatenate: `[cos(t * freq_0), ..., cos(t * freq_{d/2-1}), sin(t * freq_0), ..., sin(t * freq_{d/2-1})]`

**Key implementation details:**
- Uses `float64` internally (via `position.type(torch.float64)`) to avoid precision loss in the outer product
- Returns the input dtype at the end (`.to(position.dtype)`) -- so if you pass `float32` positions, you get `float32` embeddings back
- `dim` must be even (since it splits into cos and sin halves of size `dim//2`)

Source: `diffsynth/models/wan_video_dit.py`, line 68

In [5]:
# Source: diffsynth/models/wan_video_dit.py, line 68
freq_dim = 256
timesteps = torch.arange(50, dtype=torch.float32)
emb = sinusoidal_embedding_1d(freq_dim, timesteps)
assert emb.shape == torch.Size([50, freq_dim]), f"Expected (50, {freq_dim}), got {emb.shape}"
print(f"Input:  {timesteps.shape}  (integer timesteps)")
print(f"Output: {emb.shape}  (continuous embeddings)")
print(f"Input dtype: {timesteps.dtype}, Output dtype: {emb.dtype}")
print(f"First row (t=0): min={emb[0].min():.4f}, max={emb[0].max():.4f}")

Input:  torch.Size([50])  (integer timesteps)
Output: torch.Size([50, 256])  (continuous embeddings)
Input dtype: torch.float32, Output dtype: torch.float32
First row (t=0): min=0.0000, max=1.0000


In [ ]:
# Explore the internal structure: first half = cosine, second half = sine
cos_half = emb[:, :freq_dim//2]
sin_half = emb[:, freq_dim//2:]
assert cos_half.shape == torch.Size([50, freq_dim//2])
assert sin_half.shape == torch.Size([50, freq_dim//2])
print(f"Cosine half: {cos_half.shape}")
print(f"Sine half:   {sin_half.shape}")
# At t=0, cos(0)=1 for all frequencies, sin(0)=0
print(f"t=0 cosine values (first 5): {cos_half[0, :5]}")
print(f"t=0 sine values (first 5):   {sin_half[0, :5]}")
assert (cos_half[0, :5] - 1.0).abs().max() < 1e-4, "cos(0) should be ~1"
assert sin_half[0, :5].abs().max() < 1e-4, "sin(0) should be ~0"
print("Structure verified: cos half starts at 1.0, sin half starts at 0.0")

## 3. The Modulate Function

The `modulate` function is the core of adaptive layer normalization (adaLN) conditioning. It applies an affine transformation to a tensor using externally provided `scale` and `shift` values:

```
modulate(x, shift, scale) = x * (1 + scale) + shift
```

**Key design point -- scale as an additive offset:**
The scale is applied as `1 + scale`, not directly as `scale`. This means:
- When `scale = 0`: the multiplier is `1 + 0 = 1`, so `x * 1 = x` (no scaling effect)
- When `scale = 1`: the multiplier is `1 + 1 = 2`, so `x * 2` (doubles the activations)

This is the **identity initialization property**: if both `scale=0` and `shift=0`, then `modulate(x, 0, 0) = x * (1 + 0) + 0 = x`. The output equals the input exactly. This property is critical for training stability in adaLN-Zero (covered in NB-05): the gates start near zero, so each DiT block starts as an identity transformation.

**Practical usage:**
In WAN 2.2, `scale` and `shift` are produced by a learned linear projection from the timestep embedding plus a per-block learned offset parameter. They have shape `[B, 1, dim]` -- the `1` broadcasts over the sequence dimension `S`.

Source: `diffsynth/models/wan_video_dit.py`, line 64

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 64
B, S, dim = 1, 10, 384
x = torch.randn(B, S, dim)
scale = torch.zeros(B, 1, dim)
shift = torch.zeros(B, 1, dim)
out = modulate(x, shift, scale)
assert out.shape == torch.Size([B, S, dim])
assert torch.allclose(out, x), "With scale=0, shift=0: output should equal input"
print("Identity verified: scale=0, shift=0 -> output == input")
print(f"Max difference: {(out - x).abs().max().item()}")
print(f"Output shape: {out.shape}")

In [ ]:
scale = torch.ones(B, 1, dim)
shift = torch.full((B, 1, dim), 0.5)
out = modulate(x, shift, scale)
assert out.shape == torch.Size([B, S, dim])
expected = 2 * x + 0.5
assert torch.allclose(out, expected), "modulate(x, 0.5, 1) should equal 2x + 0.5"
print(f"scale=1, shift=0.5: multiplier=(1+1)=2, then +0.5")
print(f"Input mean:  {x.mean():.4f}")
print(f"Output mean: {out.mean():.4f}")
print(f"Verified: output = 2x + 0.5")

## Summary

### Key Takeaways
- **RMSNorm** normalizes by root-mean-square without centering the mean -- simpler and faster than LayerNorm, with only a `weight` parameter (no bias). Upcasts to float32 internally for stability. Used throughout WAN 2.2's DiT blocks for q/k normalization and pre-norm in each block.
- **sinusoidal_embedding_1d** maps integer timestep positions to dense frequency-domain vectors using sin/cos functions. Uses float64 internally for precision. Output dimension must be even (split equally between cos and sin halves). Input dtype is preserved in the output.
- **modulate** applies affine conditioning via `x * (1 + scale) + shift`. When scale=0 and shift=0, it is the identity function -- this property is critical for adaLN-Zero (NB-05), where blocks start as identity transformations during training initialization.

### Source References
| Symbol | Location |
|--------|----------|
| RMSNorm | `diffsynth/models/wan_video_dit.py`, line 101 |
| sinusoidal_embedding_1d | `diffsynth/models/wan_video_dit.py`, line 68 |
| modulate | `diffsynth/models/wan_video_dit.py`, line 64 |

## Exercises

### Exercise 1 -- Swap normalization
Replace `RMSNorm(dim=dim)` with `nn.LayerNorm(dim)` in the RMSNorm demo cell. Run the shape verification. What changes? What stays the same? Compare the parameter counts (`norm.weight` vs `ln.weight` and `ln.bias`). Why does RMSNorm have half the parameters?

### Exercise 2 -- Frequency exploration
Modify the base frequency (10000) in `sinusoidal_embedding_1d` to 1000 by editing the source. Observe how the frequency bands change -- higher-index dimensions should oscillate faster. Why does the original transformer paper use 10000 as the base?

### Exercise 3 -- Modulate chain
Apply `modulate` twice in sequence with different scale/shift values: first with `scale_1=0.5, shift_1=1.0`, then with `scale_2=1.0, shift_2=-0.5`. Verify that the output shape is preserved. Derive the equivalent single-step scale and shift that produces the same result: `scale_eq = (1+s1)(1+s2)-1` and `shift_eq = (1+s2)*sh1 + sh2`.